# Cold Start Chat
## Chat system for addressing the cold start problem head on.
This will be using NLP to take in a user's input and parse any relevant information to then add to the users profile, e.g. they like horror films, they don't like Brad Pitt, etc.
This is to give them an immediate set of recommendations based on their profiles.

Another objective in these questions is to not ask redundant ones, e.g. if they've already said they don't like non-English films, we shouldn't ask them if they like French films.


### Libraries
- Recommendation library: LensKit
- NLP library: spaCy

In [3]:
import numpy as np
import lenskit
import pandas as pd
import spacy

print("NumPy version:", np.__version__)
print("LensKit version:", lenskit.__version__)
print("Pandas version:", pd.__version__)
print("spaCy version:", spacy.__version__)

NumPy version: 2.0.2
LensKit version: 0.14.4
Pandas version: 2.2.3
spaCy version: 3.8.2


In [4]:
nlp = spacy.load('en_core_web_sm')

In [5]:
# Initialize an empty user profile
user_profile = {
    'films_liked': [],
    'films_disliked': [],
    'genres': [],
    'actors': [],
    'directors': [],
    'languages': []
}

In [12]:
def parse_user_input(user_input):
    doc = nlp(user_input)
    genres = []
    actors = []
    
    for ent in doc.ents:
        if ent.label_ == 'WORK_OF_ART':  # Assuming genres are labeled as WORK_OF_ART
            genres.append(ent.text)
        elif ent.label_ == 'PERSON':  # Assuming actors are labeled as PERSON
            actors.append(ent.text)
    
    return genres, actors

In [13]:
def update_user_profile(user_input, user_profile):
    genres, actors = parse_user_input(user_input)
    
    user_profile['genres'].extend(genres)
    user_profile['actors'].extend(actors)
    
    # Remove duplicates
    user_profile['genres'] = list(set(user_profile['genres']))
    user_profile['actors'] = list(set(user_profile['actors']))
    
    return user_profile

In [9]:
from lenskit.datasets import ML100K
ml100k = ML100K('../ml-100k')
ratings = ml100k.ratings
ratings.head()

,user,item,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


In [ ]:
from lenskit import batch, topn
from lenskit.datasets import ML100K

# Load the dataset
ml100k = ML100K('../ml-100k')
ratings = ml100k.ratings

# Function to generate recommendations
def generate_recommendations(user_profile, ratings):
    # Filter ratings based on user profile (simplified example)
    filtered_ratings = ratings[ratings['item'].isin(user_profile['genres'])]

    # Initialize RecListAnalysis
    rla = topn.RecListAnalysis()

    # Generate top-N recommendations with suffixes for overlapping columns
    recs = rla.compute(filtered_ratings, ratings, include_missing=False).add_suffix('_r')

    return recs

# Example usage
user_input = "I like horror films and I don't like Brad Pitt."
user_profile = update_user_profile(user_input, user_profile)
recommendations = generate_recommendations(user_profile, ratings)

print(recommendations)

In [ ]:
# Questions to ask the user